# Bronze ingestion

Este notebook implementa la capa Bronze del pipeline.

Responsabilidades:

- Leer el archivo CSV original.
- Aplicar un esquema explícito.
- Ejecutar validaciones estructurales básicas.
- Agregar metadata técnica.
- Persistir el resultado como tabla Delta.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DecimalType,
    TimestampType
)


In [0]:
CATALOGO = "byma"

BRONZE_SCHEMA = "bronze"

BRONZE_TABLE = "operaciones_raw"

INPUT_FILE = "/Volumes/byma/bronze/landing/data_set_challenge.csv"

In [0]:
df_header = (
    spark.read
        .option("header", True)
        .csv(INPUT_FILE)
)

print(df_header.columns)

display(df_header.limit(10))

df_header.printSchema()

['fecha', 'tipoTran', 'id_cliente', 'descripcion_titulo', 'moneda', 'simbolo_titulo', 'cantidad', 'precio', 'id_transaccion', 'origen']


fecha,tipoTran,id_cliente,descripcion_titulo,moneda,simbolo_titulo,cantidad,precio,id_transaccion,origen
2026-01-06T20:24:51.000Z,Compra,CLI56FC291A,Cedear Nio Inc.,ARS,NIO,1,1861.7702,TXNxaji0y6dpbhsa,Sitio Web Desktop
2026-02-19T17:35:16.000Z,Compra,CLI1CC7542B,Cedear Microsoft Corp.,ARS,MSFT,24,19481.5155,TXNhxthv3a3zmf8m,Sitio Web Desktop
2026-01-28T10:46:57.000Z,Compra,CLI628A1442,Cedear Etf Spdr Gold Trust,ARS,GLD,2,14678.4659,TXNdd4v30t9nt3w5,App Mobile
2026-03-05T16:17:03.000Z,Venta,CLID8A4C08F,Cedear Microstrategy Inc,ARS,MSTR,40,10789.072,TXNuzbikcidkwnnh,API
2026-02-23T23:43:53.000Z,Compra,CLI6691FC58,Bono Rep. Argentina Usd Step Up 2030,ARS,AL30,419,856.398,TXNj7xvg0fn9xuy4,Sitio Web Desktop
2026-02-25T02:33:14.000Z,Compra,CLI0D41FC57,Cedear Proshares Short S&P500,ARS,SH,21,6553.7541,TXN1ibljh75lxo6q,App Mobile
2026-01-06T23:37:16.000Z,Venta,CLI101D7ECC,Cedear Unitedhealth Group Inc.,ARS,UNH,1,16467.6499,TXNjiujv6oh9sdbd,App Mobile
2026-01-15T07:26:24.000Z,Venta,CLI2CB1DB63,Bono Rep. Argentina Usd Step Up 2030,ARS,AL30,10,874.4085,TXNw2pcn9t84azyt,App Mobile
2026-01-12T17:35:16.000Z,Compra,CLIDD38D05A,"Cedear Amazon.Com, Inc",USD,AMZND,104,1.7629,TXNjxepq85jsg65k,App Mobile
2026-02-27T05:04:53.000Z,Venta,CLI032865E4,BONO REP ARG AJ CER V30/06/28,ARS,TZX28,24,2.9034,TXNxvf1t2tala753,App Mobile


root
 |-- fecha: string (nullable = true)
 |-- tipoTran: string (nullable = true)
 |-- id_cliente: string (nullable = true)
 |-- descripcion_titulo: string (nullable = true)
 |-- moneda: string (nullable = true)
 |-- simbolo_titulo: string (nullable = true)
 |-- cantidad: string (nullable = true)
 |-- precio: string (nullable = true)
 |-- id_transaccion: string (nullable = true)
 |-- origen: string (nullable = true)



In [0]:

spark.sql("SET TIME ZONE 'UTC'")

schema = StructType([
    StructField("fecha", TimestampType(), True),
    StructField("tipoTran", StringType(), True),
    StructField("id_cliente", StringType(), True),
    StructField("descripcion_titulo", StringType(), True),
    StructField("moneda", StringType(), True),
    StructField("simbolo_titulo", StringType(), True),
    StructField("cantidad", DecimalType(20, 6), True),
    StructField("precio", DecimalType(20, 6), True),
    StructField("id_transaccion", StringType(), True),
    StructField("origen", StringType(), True)
])

df_raw = (
    spark.read
        .option("header", True)
        .schema(schema)
        .csv(INPUT_FILE)
)

display(df_raw.limit(10))
df_raw.printSchema()

df_raw.count()

fecha,tipoTran,id_cliente,descripcion_titulo,moneda,simbolo_titulo,cantidad,precio,id_transaccion,origen
2026-01-06T20:24:51.000Z,Compra,CLI56FC291A,Cedear Nio Inc.,ARS,NIO,1.000000,1861.770200,TXNxaji0y6dpbhsa,Sitio Web Desktop
2026-02-19T17:35:16.000Z,Compra,CLI1CC7542B,Cedear Microsoft Corp.,ARS,MSFT,24.000000,19481.515500,TXNhxthv3a3zmf8m,Sitio Web Desktop
2026-01-28T10:46:57.000Z,Compra,CLI628A1442,Cedear Etf Spdr Gold Trust,ARS,GLD,2.000000,14678.465900,TXNdd4v30t9nt3w5,App Mobile
2026-03-05T16:17:03.000Z,Venta,CLID8A4C08F,Cedear Microstrategy Inc,ARS,MSTR,40.000000,10789.072000,TXNuzbikcidkwnnh,API
2026-02-23T23:43:53.000Z,Compra,CLI6691FC58,Bono Rep. Argentina Usd Step Up 2030,ARS,AL30,419.000000,856.398000,TXNj7xvg0fn9xuy4,Sitio Web Desktop
2026-02-25T02:33:14.000Z,Compra,CLI0D41FC57,Cedear Proshares Short S&P500,ARS,SH,21.000000,6553.754100,TXN1ibljh75lxo6q,App Mobile
2026-01-06T23:37:16.000Z,Venta,CLI101D7ECC,Cedear Unitedhealth Group Inc.,ARS,UNH,1.000000,16467.649900,TXNjiujv6oh9sdbd,App Mobile
2026-01-15T07:26:24.000Z,Venta,CLI2CB1DB63,Bono Rep. Argentina Usd Step Up 2030,ARS,AL30,10.000000,874.408500,TXNw2pcn9t84azyt,App Mobile
2026-01-12T17:35:16.000Z,Compra,CLIDD38D05A,"Cedear Amazon.Com, Inc",USD,AMZND,104.000000,1.762900,TXNjxepq85jsg65k,App Mobile
2026-02-27T05:04:53.000Z,Venta,CLI032865E4,BONO REP ARG AJ CER V30/06/28,ARS,TZX28,24.000000,2.903400,TXNxvf1t2tala753,App Mobile


root
 |-- fecha: timestamp (nullable = true)
 |-- tipoTran: string (nullable = true)
 |-- id_cliente: string (nullable = true)
 |-- descripcion_titulo: string (nullable = true)
 |-- moneda: string (nullable = true)
 |-- simbolo_titulo: string (nullable = true)
 |-- cantidad: decimal(20,6) (nullable = true)
 |-- precio: decimal(20,6) (nullable = true)
 |-- id_transaccion: string (nullable = true)
 |-- origen: string (nullable = true)



100000

## Data Quality Checks

In [0]:
# Duplicados por clave de negocio
duplicados = (
    df_raw
    .groupBy("id_transaccion")
    .count()
    .filter(F.col("count") > 1)
)

duplicados_count = duplicados.count()

print(f"Duplicados por id_transaccion: {duplicados_count}")

display(duplicados.limit(20))

Duplicados por id_transaccion: 0


id_transaccion,count


In [0]:
# Valores inválidos de cantidad y precio
cantidad_invalida_count = df_raw.filter(F.col("cantidad") <= 0).count()
precio_invalido_count = df_raw.filter(F.col("precio") <= 0).count()

print(f"Registros con cantidad <= 0: {cantidad_invalida_count}")
print(f"Registros con precio <= 0: {precio_invalido_count}")

Registros con cantidad <= 0: 0
Registros con precio <= 0: 0


In [0]:
# Nulos en campos críticos
campos_criticos = [
    "fecha",
    "tipoTran",
    "id_cliente",
    "simbolo_titulo",
    "cantidad",
    "precio",
    "id_transaccion",
    "origen"
]

null_counts = []

for campo in campos_criticos:
    count_nulls = df_raw.filter(F.col(campo).isNull()).count()
    null_counts.append((campo, count_nulls))

df_nulls = spark.createDataFrame(null_counts, ["campo", "nulos"])

display(df_nulls)

campo,nulos
fecha,0
tipoTran,0
id_cliente,0
simbolo_titulo,0
cantidad,0
precio,0
id_transaccion,0
origen,0


### Client identifier validation


In [0]:
cliente_invalido = (
    df_raw
    .filter(~F.col("id_cliente").rlike("^CLI[A-F0-9]{8}$"))
)

cliente_invalido_count = cliente_invalido.count()

print(f"Registros con id_cliente fuera de patrón: {cliente_invalido_count}")

display(cliente_invalido.limit(20))

Registros con id_cliente fuera de patrón: 0


fecha,tipoTran,id_cliente,descripcion_titulo,moneda,simbolo_titulo,cantidad,precio,id_transaccion,origen


## Metadata

In [0]:
PIPELINE_VERSION = "1.0.0"

df_bronze = (
    df_raw
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.lit(INPUT_FILE))
    .withColumn("_pipeline_version", F.lit(PIPELINE_VERSION))
    .withColumn("fecha_partition", F.to_date(F.col("fecha")))
)

display(df_bronze.limit(10))
df_bronze.printSchema()

fecha,tipoTran,id_cliente,descripcion_titulo,moneda,simbolo_titulo,cantidad,precio,id_transaccion,origen,_ingested_at,_source_file,_pipeline_version,fecha_partition
2026-01-06T20:24:51.000Z,Compra,CLI56FC291A,Cedear Nio Inc.,ARS,NIO,1.000000,1861.770200,TXNxaji0y6dpbhsa,Sitio Web Desktop,2026-06-28T17:16:25.039Z,/Volumes/byma/bronze/landing/data_set_challenge.csv,1.0.0,2026-01-06
2026-02-19T17:35:16.000Z,Compra,CLI1CC7542B,Cedear Microsoft Corp.,ARS,MSFT,24.000000,19481.515500,TXNhxthv3a3zmf8m,Sitio Web Desktop,2026-06-28T17:16:25.039Z,/Volumes/byma/bronze/landing/data_set_challenge.csv,1.0.0,2026-02-19
2026-01-28T10:46:57.000Z,Compra,CLI628A1442,Cedear Etf Spdr Gold Trust,ARS,GLD,2.000000,14678.465900,TXNdd4v30t9nt3w5,App Mobile,2026-06-28T17:16:25.039Z,/Volumes/byma/bronze/landing/data_set_challenge.csv,1.0.0,2026-01-28
2026-03-05T16:17:03.000Z,Venta,CLID8A4C08F,Cedear Microstrategy Inc,ARS,MSTR,40.000000,10789.072000,TXNuzbikcidkwnnh,API,2026-06-28T17:16:25.039Z,/Volumes/byma/bronze/landing/data_set_challenge.csv,1.0.0,2026-03-05
2026-02-23T23:43:53.000Z,Compra,CLI6691FC58,Bono Rep. Argentina Usd Step Up 2030,ARS,AL30,419.000000,856.398000,TXNj7xvg0fn9xuy4,Sitio Web Desktop,2026-06-28T17:16:25.039Z,/Volumes/byma/bronze/landing/data_set_challenge.csv,1.0.0,2026-02-23
2026-02-25T02:33:14.000Z,Compra,CLI0D41FC57,Cedear Proshares Short S&P500,ARS,SH,21.000000,6553.754100,TXN1ibljh75lxo6q,App Mobile,2026-06-28T17:16:25.039Z,/Volumes/byma/bronze/landing/data_set_challenge.csv,1.0.0,2026-02-25
2026-01-06T23:37:16.000Z,Venta,CLI101D7ECC,Cedear Unitedhealth Group Inc.,ARS,UNH,1.000000,16467.649900,TXNjiujv6oh9sdbd,App Mobile,2026-06-28T17:16:25.039Z,/Volumes/byma/bronze/landing/data_set_challenge.csv,1.0.0,2026-01-06
2026-01-15T07:26:24.000Z,Venta,CLI2CB1DB63,Bono Rep. Argentina Usd Step Up 2030,ARS,AL30,10.000000,874.408500,TXNw2pcn9t84azyt,App Mobile,2026-06-28T17:16:25.039Z,/Volumes/byma/bronze/landing/data_set_challenge.csv,1.0.0,2026-01-15
2026-01-12T17:35:16.000Z,Compra,CLIDD38D05A,"Cedear Amazon.Com, Inc",USD,AMZND,104.000000,1.762900,TXNjxepq85jsg65k,App Mobile,2026-06-28T17:16:25.039Z,/Volumes/byma/bronze/landing/data_set_challenge.csv,1.0.0,2026-01-12
2026-02-27T05:04:53.000Z,Venta,CLI032865E4,BONO REP ARG AJ CER V30/06/28,ARS,TZX28,24.000000,2.903400,TXNxvf1t2tala753,App Mobile,2026-06-28T17:16:25.039Z,/Volumes/byma/bronze/landing/data_set_challenge.csv,1.0.0,2026-02-27


root
 |-- fecha: timestamp (nullable = true)
 |-- tipoTran: string (nullable = true)
 |-- id_cliente: string (nullable = true)
 |-- descripcion_titulo: string (nullable = true)
 |-- moneda: string (nullable = true)
 |-- simbolo_titulo: string (nullable = true)
 |-- cantidad: decimal(20,6) (nullable = true)
 |-- precio: decimal(20,6) (nullable = true)
 |-- id_transaccion: string (nullable = true)
 |-- origen: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = false)
 |-- _source_file: string (nullable = false)
 |-- _pipeline_version: string (nullable = false)
 |-- fecha_partition: date (nullable = true)



## Persist Bronze

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS byma.bronze")

(
    df_bronze
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy("fecha_partition")
    .option("overwriteSchema", "true")
    .saveAsTable("byma.bronze.operaciones_raw")
)

VALIDACION

In [0]:
bronze_count = spark.table("byma.bronze.operaciones_raw").count()

print(f"Registros persistidos en Bronze: {bronze_count}")

display(spark.table("byma.bronze.operaciones_raw").limit(10))

Registros persistidos en Bronze: 100000


fecha,tipoTran,id_cliente,descripcion_titulo,moneda,simbolo_titulo,cantidad,precio,id_transaccion,origen,_ingested_at,_source_file,_pipeline_version,fecha_partition
2026-01-08T00:36:46.000Z,Compra,CLI99F147FF,Grupo Financiero Galicia S.A,ARS,GGAL,2.000000,8387.630400,TXNe1v2isqp49kwv,App Mobile,2026-06-27T22:01:11.245Z,/Volumes/byma/bronze/landing/data_set_challenge.csv,1.0.0,2026-01-08
2026-01-08T01:59:41.000Z,Compra,CLI1499FE7A,Ternium Argentina Sa,ARS,TXAR,50.000000,750.218400,TXNnb74x2ek3zezq,App Mobile,2026-06-27T22:01:11.245Z,/Volumes/byma/bronze/landing/data_set_challenge.csv,1.0.0,2026-01-08
2026-01-08T02:34:15.000Z,Compra,CLICD83AF92,Cedear Apple Inc.,ARS,AAPL,1.000000,20248.747100,TXN0153sflse45bg,App Mobile,2026-06-27T22:01:11.245Z,/Volumes/byma/bronze/landing/data_set_challenge.csv,1.0.0,2026-01-08
2026-01-08T04:34:00.000Z,Venta,CLI1179013D,Bono Rep. Argentina Usd Step Up 2030,ARS,AL30,450.000000,1003.148900,TXNou6qlz2rwgwaj,App Mobile,2026-06-27T22:01:11.245Z,/Volumes/byma/bronze/landing/data_set_challenge.csv,1.0.0,2026-01-08
2026-01-08T12:13:10.000Z,Compra,CLIE6D54A7A,Bono Rep. Argentina Usd Step Up 2030,USD,AL30D,1334.000000,0.671300,TXNdw79wcxz2ei92,App Mobile,2026-06-27T22:01:11.245Z,/Volumes/byma/bronze/landing/data_set_challenge.csv,1.0.0,2026-01-08
2026-01-08T11:06:53.000Z,Compra,CLIE4A0B9FA,Cedear Pan American Silver Cor,ARS,PAAS,1.000000,26379.030800,TXNgvz9aawe429om,App Mobile,2026-06-27T22:01:11.245Z,/Volumes/byma/bronze/landing/data_set_challenge.csv,1.0.0,2026-01-08
2026-01-08T09:50:59.000Z,Compra,CLIC6141084,Banco Macro,ARS,BMA,10.000000,13985.177300,TXNr08cw2lbm2qsq,Sitio Web Desktop,2026-06-27T22:01:11.245Z,/Volumes/byma/bronze/landing/data_set_challenge.csv,1.0.0,2026-01-08
2026-01-08T18:13:36.000Z,Compra,CLI134F00C3,Cedear Apple Inc.,ARS,AAPL,12.000000,20070.758200,TXNt8cq1xgbngz6w,App Mobile,2026-06-27T22:01:11.245Z,/Volumes/byma/bronze/landing/data_set_challenge.csv,1.0.0,2026-01-08
2026-01-08T00:47:50.000Z,Compra,CLIBA20C83C,"Cedear Amazon.Com, Inc",ARS,AMZN,27.000000,2576.639900,TXN6l91f4x2bfibj,App Mobile,2026-06-27T22:01:11.245Z,/Volumes/byma/bronze/landing/data_set_challenge.csv,1.0.0,2026-01-08
2026-01-08T11:41:18.000Z,Venta,CLIA68BD402,Cedear The Coca Cola Company,ARS,KO,1.000000,20703.819000,TXN2tnxhn5e0ev6x,App Mobile,2026-06-27T22:01:11.245Z,/Volumes/byma/bronze/landing/data_set_challenge.csv,1.0.0,2026-01-08


## Quality log

In [0]:
quality_results = [
    ("duplicated_id_transaccion", duplicados_count),
    ("cantidad_menor_igual_cero", cantidad_invalida_count),
    ("precio_menor_igual_cero", precio_invalido_count),
    ("cliente_fuera_patron", cliente_invalido_count)
]

df_quality_log = (
    spark.createDataFrame(quality_results, ["tipo_anomalia", "cantidad_registros"])
    .withColumn("timestamp_deteccion", F.current_timestamp())
    .withColumn("_source_file", F.lit(INPUT_FILE))
    .withColumn("_pipeline_version", F.lit(PIPELINE_VERSION))
)

display(df_quality_log)

tipo_anomalia,cantidad_registros,timestamp_deteccion,_source_file,_pipeline_version
duplicated_id_transaccion,0,2026-06-28T17:16:26.411Z,/Volumes/byma/bronze/landing/data_set_challenge.csv,1.0.0
cantidad_menor_igual_cero,0,2026-06-28T17:16:26.411Z,/Volumes/byma/bronze/landing/data_set_challenge.csv,1.0.0
precio_menor_igual_cero,0,2026-06-28T17:16:26.411Z,/Volumes/byma/bronze/landing/data_set_challenge.csv,1.0.0
cliente_fuera_patron,0,2026-06-28T17:16:26.411Z,/Volumes/byma/bronze/landing/data_set_challenge.csv,1.0.0


In [0]:
(
    df_quality_log
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("byma.bronze.calidad_log")
)

## Idempotency check

In [0]:
bronze_count = spark.table("byma.bronze.operaciones_raw").count()
raw_count = df_raw.count()

print(f"Registros fuente: {raw_count}")
print(f"Registros Bronze: {bronze_count}")

assert bronze_count == raw_count, "La cantidad de registros en Bronze no coincide con la fuente"

Registros fuente: 100000
Registros Bronze: 100000
